# Costruire un Cubo di Sintesi per la Revenue Assurance Telecom con PROC SUMMARY

## Sintesi Esecutiva

Un team di revenue assurance di un operatore wireless pre-aggrega un mese di record di fatturazione giornalieri per abbonato in un compatto cubo di sintesi, cosicché gli analisti possano approfondire i ricavi liquidati per regione, livello di piano e tipo di chiamata senza dover ri-scansionare la tabella dei fatti grezza. `PROC SUMMARY` comprime 100 record giornalieri per abbonato in un insieme multidimensionale di celle: la granularità più fine (regione x livello piano x tipo chiamata) riempie 25 delle 27 celle possibili, mentre i sotto-cubi denominati forniscono i marginali che gli analisti interrogano più spesso. In questo mese campione l'operatore ha liquidato \$222.78 nelle tre regioni attive, con Sud (\$97.44) ed Est (\$86.94) che insieme rappresentano l'83% dei ricavi e Nord che segue a \$38.40. Il sotto-cubo singolo più ricco è il servizio Voce di livello Plus (\$59.06 su 18 record), e classificare le celle per resa per record evidenzia le celle di utilizzo dati come gli obiettivi di massimo valore per un audit delle perdite (resa massima \$7.87/record). Ogni cifra qui sotto è letta direttamente dall'output eseguito.

## Fonti dei Dati

| Dataset | Righe | Granularità | Variabili Chiave |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Una riga per riepilogo di utilizzo giornaliero per abbonato | `region` (Est/Sud/Nord), `plan_tier` (Prepagato/Base/Plus), `call_type` (Voce/SMS/Dati), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Una riga per ogni cella non vuota (regione x plan_tier x call_type) | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Una riga per ogni cella dei sotto-cubi di drill denominati | `_TYPE_`, `_FREQ_`, `rev_total` |

Tutti i dati sono generati inline con `call streaminit()` / `rand()`; non sono richiesti file esterni né accesso alla rete. Questo ambiente funziona senza licenza, quindi i dataset scritti sono limitati a 100 osservazioni - lo scenario è dimensionato in modo che il cubo sia completamente popolato entro tale limite.

## Dai record grezzi di dettaglio chiamata a un cubo esplorabile

Gli operatori wireless liquidano ricavi su milioni di eventi di utilizzo giornalieri. Per permettere agli analisti di revenue assurance di rispondere a domande come *"Qual è stato il ricavo voce di livello Plus nella regione Sud il mese scorso?"* senza dover ri-scansionare la tabella dei fatti grezza ogni volta, **pre-aggreghiamo** i dati in un compatto cubo di sintesi.

`PROC SUMMARY` è lo strumento di Base SAS pensato esattamente per questo: raggruppa una tabella dei fatti piatta per una o più dimensioni `CLASS` e scrive le statistiche richieste in un dataset di output, contrassegnando ogni riga con `_TYPE_` (quali dimensioni sono attive) e `_FREQ_` (i record dietro la cella). Quel dataset di output *è* un cubo multidimensionale - la stessa struttura di aggregazione che uno strumento OLAP esporrebbe, materializzata come un normale dataset SAS che si può stampare, unire e sezionare.

Questo notebook:

1. Genera un mese realistico di record di fatturazione giornalieri per abbonato.
2. Costruisce il cubo a granularità più fine (regione x livello piano x tipo chiamata) con `PROC SUMMARY NWAY`.
3. Materializza i **sotto-cubi di drill** denominati con l'istruzione `TYPES`.
4. Proietta i ricavi sulla base abbonati con un `WEIGHT`, approfondisce in una regione, e classifica le celle per resa per record per il triage delle perdite.

## Passo 1 - Generare dati sintetici di dettaglio chiamata / fatturazione

Ogni riga riepiloga l'utilizzo di un servizio da parte di un abbonato in un giorno. Usiamo `call streaminit` per la riproducibilità e `rand()` per estrarre distribuzioni plausibili: ricavi e utilizzo scalano con il livello di piano, il ricavo voce segue i minuti fatturabili, e il ricavo dati segue i megabyte. Ogni `RAND('table', ...)` riceve una probabilità per categoria cosicché ogni regione, livello e tipo di chiamata compaia nel campione di 100 record. Un piccolo peso di indagine `subscriber_wt` viene associato così da poter dimostrare più avanti una misura ponderata.

In [1]:
DATI work.cdr_billing;
    CHIAMARE streaminit(20260131);
    LUNGHEZZA region $6 plan_tier $9 call_type $7 device_class $14 bill_month $7;
    ETICHETTA revenue       = "Ricavi Liquidati (USD)"
          call_minutes  = "Minuti Voce Fatturabili"
          data_mb       = "Volume Dati (MB)"
          subscriber_wt = "Peso di Indagine Abbonato";

    FARE i = 1 FINO_A 100;
        /* --- Dimensioni: una probabilità per livello, che sommano a 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        SELEZIONARE (r);
            QUANDO (1) region = "Est";
            QUANDO (2) region = "Sud";
            ALTRIMENTI_S region = "Nord";
        FINE;

        p = rand("table", 0.30, 0.40, 0.30);
        SELEZIONARE (p);
            QUANDO (1) plan_tier = "Prepagato";
            QUANDO (2) plan_tier = "Base";
            ALTRIMENTI_S plan_tier = "Plus";
        FINE;

        c = rand("table", 0.50, 0.30, 0.20);
        SELEZIONARE (c);
            QUANDO (1) call_type = "Voce";
            QUANDO (2) call_type = "SMS";
            ALTRIMENTI_S call_type = "Dati";
        FINE;

        SE_COND rand("uniform") < 0.55 ALLORA device_class = "Smart";
        ALTRIMENTI device_class = "Tradizionale";

        bill_month = "2026-01";

        /* --- Misure, scalate per livello e servizio --- */
        tier_mult = (plan_tier = "Prepagato")*0.7
                  + (plan_tier = "Base")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Voce")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Dati")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        USCITA;
    FINE;
    RIMUOVERE i r p c tier_mult base_rev;
ESEGUIRE;

PROCEDURA STAMPARE DATI=work.cdr_billing(obs=8) ETICHETTA noobs;
    TITOLO "Esempio di Record di Fatturazione Giornalieri per Abbonato";
ESEGUIRE;


                               Esempio di Record di Fatturazione Giornalieri per Abbonato                               

region  plan_tier  call_type  device_class  bill_month  Minuti Voce Fatturabili  Volume Dati (MB)  Ricavi Liquidati (USD)  Peso di Indagine Abbonato
Nord    Plus       SMS        Smart         2026-01                           0                 0                    0.67                       1.13
Sud     Prepagato  SMS        Tradizionale  2026-01                           0                 0                    0.94                       1.42
Sud     Plus       SMS        Smart         2026-01                           0                 0                    0.99                       0.86
Sud     Plus       SMS        Smart         2026-01                           0                 0                    1.01                       1.03
Sud     Plus       Voce       Smart         2026-01                       103.4                 0                    5.79            


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Passo 2 - Costruire il cubo a granularità più fine con PROC SUMMARY NWAY

`NWAY` mantiene solo la combinazione più dettagliata di tutte le dimensioni `CLASS` - qui ogni cella popolata (regione x plan_tier x call_type). Per ogni cella memorizziamo `SUM`, `MEAN` e `MAX` del ricavo più il totale dei minuti e dei megabyte, cosicché un analista possa leggere il ricavo totale per cella, derivare la media per record, e individuare l'addebito singolo più grande. `_FREQ_` registra quanti giorni-abbonato ci sono dietro ogni cella. Qui eliminiamo `_TYPE_` perché, alla granularità NWAY, ogni riga ha lo stesso tipo.

In [2]:
PROCEDURA summary DATI=work.cdr_billing NWAY;
    CLASSE region plan_tier call_type;
    VARIABILE revenue call_minutes data_mb;
    USCITA out=work.cube_nway(RIMUOVERE=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
ESEGUIRE;

PROCEDURA STAMPARE DATI=work.cube_nway(obs=12) noobs;
    TITOLO "Celle del cubo NWAY (regione * livello_piano * tipo_chiamata)";
    FORMATO rev_total rev_mean rev_max min_total data_total comma10.2;
ESEGUIRE;


                             Celle del cubo NWAY (regione * livello_piano * tipo_chiamata)                              

REGION  PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL  REV_MEAN  REV_MAX  MIN_TOTAL  DATA_TOTAL
Est     Base       Dati            4      18.17      4.54     8.05       0.00    1,807.90
Est     Base       SMS             4       4.07      1.02     1.24       0.00        0.00
Est     Base       Voce            7      15.08      2.15     3.73     302.50        0.00
Est     Plus       Dati            1       5.54      5.54     5.54       0.00      573.90
Est     Plus       SMS             4       3.59      0.90     0.95       0.00        0.00
Est     Plus       Voce            7      23.87      3.41     8.01     491.70        0.00
Est     Prepagato  SMS             3       3.00      1.00     1.06       0.00        0.00
Est     Prepagato  Voce            8      13.62      1.70     6.50     270.20        0.00
Nord    Base       Dati            1       7.87      7.87     7.87  


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Passo 3 - Materializzare i sotto-cubi di drill denominati con TYPES

Un cubo OLAP pre-memorizza le aggregazioni che gli analisti navigano più spesso. L'istruzione `TYPES` fa esattamente questo: ogni termine chiede a `PROC SUMMARY` di emettere un sotto-cubo. Richiediamo il totale generale `()`, il marginale `region`, e i sotto-cubi a due vie `region*plan_tier` e `call_type*plan_tier` - i percorsi di drill che una dashboard di ricavi esporrebbe. SAS contrassegna ogni sotto-cubo con un codice `_TYPE_` (una bitmask sulla lista `CLASS`), cosicché un unico dataset di output porti ogni livello della gerarchia.

In [3]:
PROCEDURA summary DATI=work.cdr_billing;
    CLASSE region plan_tier call_type;
    VARIABILE revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    USCITA out=work.cube_hier
           sum(revenue)=rev_total;
ESEGUIRE;

PROCEDURA STAMPARE DATI=work.cube_hier noobs;
    TITOLO "Aggregazioni dei sotto-cubi: totale generale, regione, regione*livello, tipo_chiamata*livello";
    VARIABILE _type_ region plan_tier call_type _freq_ rev_total;
    FORMATO rev_total comma10.2;
ESEGUIRE;


             Aggregazioni dei sotto-cubi: totale generale, regione, regione*livello, tipo_chiamata*livello              

_TYPE_  REGION  PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL
     0                                   100     222.78
     3          Base       Dati            8      38.06
     3          Base       SMS             8       8.03
     3          Base       Voce           20      42.33
     3          Plus       Dati            3      17.46
     3          Plus       SMS            13      11.75
     3          Plus       Voce           18      59.06
     3          Prepagato  Dati            7      14.50
     3          Prepagato  SMS             7       6.82
     3          Prepagato  Voce           16      24.77
     4  Est                               38      86.94
     4  Nord                              23      38.40
     4  Sud                               39      97.44
     6  Est     Base                      15      37.32
     6  Est     Plus                  


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Passo 4 - Proiezione ponderata, un drill regionale, e triage delle perdite

Tre letture che un team di revenue assurance esegue realmente sul cubo:

- **Proiezione ponderata.** Applicare `WEIGHT=subscriber_wt` a una sintesi `region*plan_tier` riporta i ricavi scalati sull'intera base abbonati che il campione rappresenta, anziché sulle righe campionate grezze.
- **Drill.** Filtrare il cubo NWAY su una regione espande un singolo ramo della gerarchia - qui Sud - al suo dettaglio livello-per-servizio.
- **Triage delle perdite.** Ordinare le celle per ricavo medio per record evidenzia le celle a resa più alta; le celle a bassa frequenza e alta resa sono esattamente ciò che un audit esamina per ricavi mal tariffati o in perdita.

In [4]:
/* Ricavi ponderati proiettati sulla base abbonati */
PROCEDURA summary DATI=work.cdr_billing NWAY;
    CLASSE region plan_tier;
    VARIABILE revenue;
    PESO subscriber_wt;
    USCITA out=work.cube_wt(RIMUOVERE=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
ESEGUIRE;

PROCEDURA STAMPARE DATI=work.cube_wt noobs;
    TITOLO "Ricavi ponderati per regione * livello piano (proiettati sulla base abbonati)";
    FORMATO rev_weighted comma10.2;
ESEGUIRE;

/* Approfondimento nel ramo della regione Sud del cubo */
PROCEDURA STAMPARE DATI=work.cube_nway noobs;
    DOVE region = "Sud";
    TITOLO "Approfondimento su Sud: celle di ricavo per livello e tipo di chiamata";
    VARIABILE plan_tier call_type _freq_ rev_total rev_mean;
    FORMATO rev_total rev_mean comma10.2;
ESEGUIRE;

/* Classifica le celle per resa per record per il triage delle perdite */
PROCEDURA ORDINARE DATI=work.cube_nway out=work.cube_ranked;
    PER DISCENDENTE rev_mean;
ESEGUIRE;

PROCEDURA STAMPARE DATI=work.cube_ranked(obs=6) noobs;
    TITOLO "Celle con ricavo medio più alto (resa per record)";
    VARIABILE region plan_tier call_type _freq_ rev_mean rev_max;
    FORMATO rev_mean rev_max comma10.2;
ESEGUIRE;


                     Ricavi ponderati per regione * livello piano (proiettati sulla base abbonati)                      

REGION  PLAN_TIER  REV_WEIGHTED  RECORDS
Est     Base              50.85       15
Est     Plus              59.59       12
Est     Prepagato         29.77       11
Nord    Base              18.30        7
Nord    Plus              22.42        7
Nord    Prepagato         20.56        9
Sud     Base              58.63       14
Sud     Plus              56.29       15
Sud     Prepagato         27.77       10

                         Approfondimento su Sud: celle di ricavo per livello e tipo di chiamata                         

PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL  REV_MEAN
Base       Dati            3      12.02      4.01
Base       SMS             2       2.01      1.00
Base       Voce            9      22.51      2.50
Plus       Dati            2      11.92      5.96
Plus       SMS             5       5.16      1.03
Plus       Voce            8      27.07      


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Interpretazione dei risultati

Il cubo di sintesi trasforma 100 righe grezze per abbonato-giorno in un compatto insieme di celle pre-aggregate che rispondono istantaneamente alle domande di drill-down, senza ri-scansionare la tabella dei fatti:

- **Dove risiedono i ricavi.** Il marginale `region` (`_TYPE_=4`) mostra che Sud ha liquidato \$97.44 ed Est \$86.94 - insieme l'83% del mese da \$222.78 - mentre Nord ha contribuito con \$38.40. All'interno del sotto-cubo `call_type*plan_tier` (`_TYPE_=3`), Voce di livello Plus è la cella singola più ricca con \$59.06 su 18 record, seguita da Voce di livello Base con \$42.33.
- **Proiezione ponderata.** Applicare il peso di indagine ridistribuisce la classifica verso i piani i cui abbonati portano più peso: Est-Plus (\$59.59) e Sud-Base (\$58.63) guidano il ricavo proiettato `region*plan_tier`, un quadro diverso dai totali regionali non ponderati e un promemoria di riportare il ricavo proiettato anziché quello campionato quando si dimensiona l'esposizione.
- **Resa per record e triage delle perdite.** Classificare le celle NWAY per `rev_mean` mette in cima le celle di utilizzo dati - Nord-Base-Dati (\$7.87/record) e Sud-Plus-Dati (\$5.96/record) - confermando che l'utilizzo intensivo di dati genera il ricavo per record più alto. I leader a bassa frequenza (uno o due record) sono esattamente le celle per cui un analista di revenue assurance andrebbe a recuperare i record di dettaglio chiamata sottostanti, per confermare che l'addebito elevato sia correttamente tariffato e non un errore.

Per un team di revenue assurance, questo cubo è il fondamento per le dashboard di variance: confrontare i ricavi liquidati con i ricavi attesi da tariffario per ogni cella (regione, livello, tipo chiamata), e le celle il cui totale medio o ponderato devia di più diventano i casi di perdita da sottoporre ad audit. Poiché l'intera struttura è un normale dataset SAS, il cubo del mese successivo può essere unito, differenziato, o congiunto con un tariffario usando gli stessi strumenti di Base SAS.